In [ ]:
# Cell 1：安装必要依赖（不碰 torch，避免降级）
!pip install -q transformers peft accelerate gguf sentencepiece protobuf
print("✅ 依赖安装完成")

In [ ]:
# Cell 2：确认 LoRA adapter 路径
import os

ADAPTER_PATH = "/kaggle/input/datasets/lin07077/lora-adapter/lora_adapter"

if os.path.exists(ADAPTER_PATH):
    print(f"✅ Adapter 已找到：{ADAPTER_PATH}")
    print(os.listdir(ADAPTER_PATH))
else:
    print(f"❌ 路径不存在，正在搜索 adapter_config.json ...")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files:
            print(f"  → 找到：{root}")
    print("请将上面找到的路径填入 ADAPTER_PATH")

In [ ]:
# Cell 3：合并 LoRA adapter（float16，CPU 模式）
# P100 CUDA kernel 与当前 PyTorch 版本不兼容（sm_60），改用 CPU merge
# Kaggle P100 机器 CPU RAM ≈ 29GB，完全能装下 7B float16（约 14GB）
import torch, os, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT_DIR    = "/kaggle/working/merged_model"

print("📥 加载基础模型（float16 + CPU）...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu",
    low_cpu_mem_usage=True,   # 逐层加载，降低峰值内存
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

print("🔗 加载并合并 LoRA adapter...")
model  = PeftModel.from_pretrained(base, ADAPTER_PATH)
merged = model.merge_and_unload()

print("💾 保存合并后模型...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
merged.save_pretrained(OUTPUT_DIR, safe_serialization=True, max_shard_size="5GB")
tokenizer.save_pretrained(OUTPUT_DIR)

# ⚠️ 清除 quantization_config（若存在会导致 llama.cpp 转换报错）
cfg_path = os.path.join(OUTPUT_DIR, "config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
if cfg.pop("quantization_config", None):
    with open(cfg_path, "w") as f:
        json.dump(cfg, f, indent=2)
    print("🧹 已清除 quantization_config")

print("✅ 合并完成，模型已保存至", OUTPUT_DIR)

In [ ]:
# Cell 4：克隆 llama.cpp（只装 GGUF 转换所需依赖，不碰 torch）
!git clone https://github.com/ggerganov/llama.cpp --depth=1
!pip install -q gguf sentencepiece protobuf
print("✅ llama.cpp 克隆完成")

In [ ]:
# Cell 5：转换 GGUF f16 → 量化为 Q4_K_M
# 磁盘策略：
#   /kaggle/working  20GB 限制，merged_model 已占 14GB
#   /tmp             RAM 虚拟盘（29GB RAM 此时大部分空闲），写 15GB f16 临时文件
#   流程：f16 写 /tmp → 删 merged_model（释放 14GB）→ 量化 Q4 写 /kaggle/working → 删 f16
import subprocess, os, shutil

OUTPUT_DIR = "/kaggle/working/merged_model"
GGUF_F16   = "/tmp/chef_lora_f16.gguf"          # 写 RAM 盘，不占 working 空间
GGUF_Q4    = "/kaggle/working/chef_lora_q4.gguf"

# Step A：HuggingFace → GGUF f16（输出到 /tmp）
print("🔄 转换为 GGUF f16（写入 /tmp）...")
result = subprocess.run(
    ["python", "llama.cpp/convert_hf_to_gguf.py", OUTPUT_DIR,
     "--outfile", GGUF_F16, "--outtype", "f16"],
    capture_output=True, text=True
)
print(result.stdout[-2000:])   # 只打印末尾避免刷屏
if result.returncode != 0:
    print("❌ 转换失败：")
    print(result.stderr[-2000:])
    raise RuntimeError("GGUF 转换失败")
print(f"✅ GGUF f16：{os.path.getsize(GGUF_F16)/1e9:.1f} GB")

# Step B：删除 merged_model，释放 /kaggle/working 空间
print("🧹 删除 merged_model，释放磁盘...")
shutil.rmtree(OUTPUT_DIR)
print(f"   /kaggle/working 剩余空间已释放")

# Step C：编译 llama-quantize
print("🔨 编译 llama-quantize...")
subprocess.run(["cmake", "llama.cpp", "-B", "llama.cpp/build"], check=True)
subprocess.run(
    ["cmake", "--build", "llama.cpp/build", "--config", "Release",
     "-j4", "--target", "llama-quantize"],
    check=True
)

# Step D：Q4_K_M 量化（从 /tmp 读，写入 /kaggle/working）
print("⚙️ 量化为 Q4_K_M...")
subprocess.run(
    ["./llama.cpp/build/bin/llama-quantize", GGUF_F16, GGUF_Q4, "Q4_K_M"],
    check=True
)

# Step E：清理 /tmp 临时文件
os.remove(GGUF_F16)

size_gb = os.path.getsize(GGUF_Q4) / 1e9
print(f"✅ 量化完成：chef_lora_q4.gguf（{size_gb:.1f} GB）")
print("📂 Kaggle 右侧 Output 面板 → /kaggle/working/chef_lora_q4.gguf → 点击下载")

In [ ]:
# Cell 6：下载提示
# Kaggle 不支持 google.colab.files.download()
# 直接在右侧「Output」面板找到 chef_lora_q4.gguf，点击下载即可
print("✅ 全部完成！")
print("👉 下载方式：Kaggle Notebook 右侧 Output 面板 → /kaggle/working/chef_lora_q4.gguf → 点击下载")